# nbtools

> Read, grep, and edit Jupyter notebook cells directly (by cell id), without hand-rolling JSON surgery each time. Plain functions -- import them directly, drive them from the `slmn` CLI, or expose them over MCP (see `03_mcp.ipynb`).

In [ ]:
#| default_exp nbtools

In [ ]:
#| export
import json, re, os
from pathlib import Path
import nbformat as _nbf

## Path safety

Every path-taking tool below first calls `_resolve_safe`. It's a no-op by default, but if
`SLMN_SANDBOX_ROOTS` is set (colon-separated directories) it refuses any path that resolves
outside them -- a guardrail for when these tools are driven by an agent.

In [ ]:
#| export
def _resolve_safe(path:str) -> str:
    "Resolve `path`, enforcing SLMN_SANDBOX_ROOTS if set (colon-separated allowed directories, e.g. '/home/user/notebooks:/tmp'). Off by default: with no env var, any path is allowed, same as before sandboxing existed. Raises PermissionError if the resolved path falls outside every allowed root. Every path-taking tool in this module calls this first."
    resolved = Path(path).resolve()
    roots = os.environ.get('SLMN_SANDBOX_ROOTS')
    if not roots: return str(resolved)
    allowed = [Path(r).resolve() for r in roots.split(':') if r]
    if not any(resolved.is_relative_to(root) for root in allowed):
        raise PermissionError(f"{path!r} resolves outside SLMN_SANDBOX_ROOTS ({roots})")
    return str(resolved)

## Reading & searching

`read_nb` and `grep_nb` are the read side. `read_nb` with no arguments lists every cell's
id and a one-line preview -- the usual way to get your bearings before a targeted read or
edit, since every other tool here addresses cells *by id*.

In [ ]:
#| export
def read_nb(path:str, # path to the .ipynb file
            cell_ids:list[str]=None, # if given, only these cells' full source (ids may be partial/substring matches)
            full:bool=False # if True (and no cell_ids given), print full source of every cell
            ) -> str:
    "Read cell source(s) from a Jupyter notebook. With no cell_ids and full=False, lists every cell's id + a source preview (one line each) -- good for getting your bearings before a targeted read."
    path = _resolve_safe(path)
    nb = json.load(open(path))
    cell_ids = cell_ids or []
    out = []
    for cell in nb['cells']:
        cid = cell.get('id', '')
        src = ''.join(cell['source'])
        if full or (cell_ids and any(q in cid for q in cell_ids)):
            out.append(f"=== {cid} ===\n{src}\n")
        elif not cell_ids:
            out.append(f"{cid}  {src[:60].replace(chr(10), ' ')}")
    return '\n'.join(out)

In [ ]:
#| export
def grep_nb(path:str, # path to the .ipynb file
            pattern:str, # regex pattern to search for, per line, within each cell's source
            context:int=0, # lines of context before/after each match to include
            line_numbers:bool=False # prefix each shown line with its (0-based) line number within the cell
            ) -> str:
    "Grep cell sources in a Jupyter notebook. Returns the matching cells (id header + matching/context lines); empty string if nothing matched."
    path = _resolve_safe(path)
    nb = json.load(open(path))
    blocks = []
    for cell in nb['cells']:
        cid = cell.get('id', '')
        src = ''.join(cell.get('source', []))
        lines = src.split('\n')
        match_indices = [i for i, l in enumerate(lines) if re.search(pattern, l)]
        if not match_indices: continue
        shown = set()
        cell_lines = [f"=== {cid} ==="]
        for mi in match_indices:
            lo, hi = max(0, mi - context), min(len(lines) - 1, mi + context)
            if lo > 0 and shown and min(shown) > lo + 1:
                cell_lines.append('--')
            for li in range(lo, hi + 1):
                if li not in shown:
                    prefix = f"{li:4d}: " if line_numbers else ""
                    marker = ">" if li == mi else " "
                    cell_lines.append(f"{marker}{prefix}{lines[li]}")
                    shown.add(li)
        blocks.append('\n'.join(cell_lines))
    return '\n\n'.join(blocks)

## Editing cells

Two ways to change an existing cell, both addressed by id: `edit_nb` does a find/replace
within a cell's source (like a code editor's replace), while `patch_nb_cell` overwrites a
cell's source wholesale. All writes use `json.dump(..., indent=1)` to match Jupyter's own
formatting, so a one-cell change stays a one-cell git diff.

In [ ]:
#| export
def edit_nb(path:str, # path to the .ipynb file
            cell_id:str, # cell id, or a prefix of one
            old_str:str, # text to find in the cell's source (first occurrence)
            new_str:str # replacement text
            ) -> str:
    "Replace the first occurrence of old_str with new_str in one cell's source, addressed by id-prefix. Falls back to a trailing-whitespace-insensitive match if the exact text isn't found. Raises ValueError if the cell or the text can't be found."
    path = _resolve_safe(path)
    nb = json.load(open(path))
    cell = next((c for c in nb['cells'] if c.get('id', '').startswith(cell_id)), None)
    if cell is None:
        raise ValueError(f"cell {cell_id!r} not found in {path}")
    src = ''.join(cell['source'])
    if old_str not in src:
        strip_trailing = lambda s: '\n'.join(l.rstrip() for l in s.split('\n'))
        src_stripped, old_stripped = strip_trailing(src), strip_trailing(old_str)
        if old_stripped not in src_stripped:
            raise ValueError(f"old_string not found in cell {cell_id}")
        src, old_str = src_stripped, old_stripped
    cell['source'] = src.replace(old_str, new_str, 1)
    json.dump(nb, open(path, 'w'), indent=1, ensure_ascii=False)
    return f"OK: updated cell {cell_id} in {path}"

In [ ]:
#| export
def patch_nb_cell(path:str, # path to the .ipynb file
                   cell_id:str, # exact cell id
                   new_source:str # the cell's complete new source, replacing the old source entirely
                   ) -> str:
    "Replace a cell's entire source (exact id match, not a prefix -- use this when you want to overwrite a cell wholesale rather than patch part of it; see edit_nb for a targeted string replacement). Raises ValueError if the cell isn't found."
    path = _resolve_safe(path)
    nb = json.load(open(path))
    cell = next((c for c in nb['cells'] if c.get('id') == cell_id), None)
    if cell is None:
        raise ValueError(f"cell {cell_id} not found in {path}")
    cell['source'] = new_source
    json.dump(nb, open(path, 'w'), indent=1, ensure_ascii=False)
    return f"OK: patched cell {cell_id} in {path}"

## Adding cells

`add_nb_cell` is the single-cell primitive; `insert_cells` is a thin loop over it for bulk
inserts. `add_nb_cell` -- not the caller -- owns the `#| export` / `#| eval: false`
directives for code cells: pass plain source plus the `export`/`eval` flags and it
regenerates them idempotently, while leaving other directives (`#| default_exp`, `#| hide`)
alone. (This very notebook edit was made with these tools.)

In [ ]:
#| export
def insert_cells(path:str, # path to the .ipynb file
                  anchor_id:str, # exact cell id to insert after
                  sources:list[str], # one or more new cells' source, inserted in order right after the anchor -- plain source, no '#| ...' directives (see add_nb_cell)
                  export:bool=True, # forwarded to add_nb_cell for every inserted cell -- see there
                  eval:bool=True # forwarded to add_nb_cell for every inserted cell -- see there
                  ) -> str:
    "Insert one or more new code cells immediately after the cell with id `anchor_id`. A thin bulk convenience wrapper over add_nb_cell (the single-cell primitive), so the actual nbformat plumbing -- including the '#| export'/'#| eval: false' directive handling -- lives in one place. Raises ValueError if the anchor isn't found."
    last = anchor_id
    for s in sources:
        last = add_nb_cell(path, s, cell_type='code', after_id=last, export=export, eval=eval)
    return f"OK: inserted {len(sources)} cell(s) after {anchor_id} in {path}"

In [ ]:
#| export
# Only these two directives are managed by add_nb_cell's export/eval flags; any other leading
# directive a caller includes (e.g. '#| default_exp', '#| hide', '#| echo: false') is preserved.
_MANAGED_DIRECTIVE_RE = re.compile(r'^[ \t]*#\|[ \t]*(?:export|eval:[ \t]*false)[ \t]*(?:\n|$)')

def _strip_managed_directives(source:str) -> str:
    "Remove leading '#| export' / '#| eval: false' lines (the two add_nb_cell regenerates from its flags), leaving every other line -- including other '#| ...' directives -- untouched."
    while _MANAGED_DIRECTIVE_RE.match(source):
        source = _MANAGED_DIRECTIVE_RE.sub('', source, count=1)
    return source

def add_nb_cell(path:str, # path to the .ipynb file
                source:str, # the new cell's source text. Don't include '#| export'/'#| eval: false' -- they're regenerated from the flags below (any you do include are stripped first, so it stays idempotent). OTHER directives like '#| default_exp' or '#| hide' ARE preserved, so pass those literally.
                cell_type:str='code', # nbformat cell type: 'code', 'markdown', or 'raw'
                after_id:str=None, # id of the cell to insert immediately after; None appends at the end (in cell order, NOT a raw file append)
                cell_id:str=None, # if given, force the new cell's id (e.g. to match an intended nbdev '# %%' marker); default lets nbformat assign one
                export:bool=True, # code cells only: prepend '#| export' so nbdev exports this cell into the module. Ignored for markdown/raw cells.
                eval:bool=True # code cells only: if False, prepend '#| eval: false' so nbdev skips executing this cell during tests/docs (e.g. a live-server/network demo). Ignored for markdown/raw cells.
                ) -> str:
    "Add a new cell to a Jupyter notebook and return its id. Appends to the end by default, or inserts right after `after_id`. `cell_type` is an nbformat type ('code'/'markdown'/'raw') -- note that boopiter-style note/prompt/assistant cells are all 'markdown' at the nbformat level. This tool -- not the caller -- owns the '#| export'/'#| eval: false' directives for code cells: pass plain source and use `export`/`eval` (any of those two you include literally get stripped and regenerated, so it's idempotent; other directives like '#| default_exp'/'#| hide' are left as-is). Reads/writes the file as plain JSON (indent=1) like the other nbtools here, so a one-cell add stays a one-cell git diff rather than reserializing the whole notebook. Raises ValueError on an unknown `cell_type`, or an `after_id` that isn't present."
    path = _resolve_safe(path)
    nb = json.load(open(path))
    makers = {'code': _nbf.v4.new_code_cell, 'markdown': _nbf.v4.new_markdown_cell, 'raw': _nbf.v4.new_raw_cell}
    if cell_type not in makers:
        raise ValueError(f"cell_type must be one of {sorted(makers)}, got {cell_type!r}")
    if cell_type == 'code':
        directives = [d for d, on in (('#| export', export), ('#| eval: false', not eval)) if on]
        source = '\n'.join(directives + [_strip_managed_directives(source)])
    cell = dict(makers[cell_type](source))  # nbformat builds a schema-valid cell (id/metadata/etc.); keep it as a plain dict
    if cell_id is not None: cell['id'] = cell_id
    cells = nb['cells']
    if after_id is None:
        cells.append(cell)
    else:
        idx = next((i for i, c in enumerate(cells) if c.get('id') == after_id), None)
        if idx is None:
            raise ValueError(f"after_id {after_id!r} not found in {path}")
        cells.insert(idx + 1, cell)
    json.dump(nb, open(path, 'w'), indent=1, ensure_ascii=False)
    return cell['id']

## Writing docs in bulk

`write_nb_docs` is the batch form of the two tools above, for documenting a notebook in one
shot: give it `notes` (markdown section headings to insert after named cells) and/or
`patches` (cells to rewrite), and it validates every id *before* writing anything -- so a
single wrong id fails cleanly instead of leaving a half-documented notebook. It's the
reusable version of this very notebook's doc pass; point it at another project to do the
same there.

In [ ]:
#| export
def write_nb_docs(path:str, # path to the .ipynb file
               notes:list=None, # markdown Note cells to INSERT: a list of [after_id, source] pairs. Each cell is inserted right after the cell whose id equals after_id; order within the list is preserved even when several notes share one anchor.
               patches:list=None, # existing cells to REWRITE: a list of [cell_id, new_source] pairs. Each named cell's entire source is replaced wholesale (like patch_nb_cell).
               cell_type:str='markdown' # nbformat cell type for the inserted `notes` ('markdown' for docs; 'code'/'raw' also accepted)
               ) -> str:
    "Apply a batch of documentation edits to a notebook in ONE atomic pass: insert markdown section-Note cells after named anchor cells (`notes`) and/or rewrite existing cells wholesale (`patches`). Every referenced id is validated against the notebook FIRST, so a single wrong id raises before anything is written -- no half-applied state (the failure mode of firing off many add_nb_cell/patch_nb_cell calls in a row, where one bad id leaves the earlier ones already committed). The whole notebook is written once with json.dump(indent=1), so it stays a minimal git diff. This is the reusable form of the 'add section headings throughout a notebook, then tidy the index' workflow -- point it at any nbdev notebook (slmn's own, or another project like boopiter) to document it. `notes` source is inserted verbatim (markdown needs no directives); for code cells that need '#| export'/'#| eval' use add_nb_cell instead."
    path = _resolve_safe(path)
    nb = json.load(open(path))
    notes = notes or []
    patches = patches or []
    makers = {'code': _nbf.v4.new_code_cell, 'markdown': _nbf.v4.new_markdown_cell, 'raw': _nbf.v4.new_raw_cell}
    if cell_type not in makers:
        raise ValueError(f"cell_type must be one of {sorted(makers)}, got {cell_type!r}")
    cells = nb['cells']
    # validate every id up front -- atomic: nothing is written unless all ids resolve
    ids = {c.get('id') for c in cells}
    missing = [a for a, _ in notes if a not in ids] + [cid for cid, _ in patches if cid not in ids]
    if missing:
        raise ValueError(f"cell id(s) not found in {path}: {missing}")
    # rewrites first -- id-addressed, so unaffected by the inserts below
    for cid, new_source in patches:
        next(c for c in cells if c.get('id') == cid)['source'] = new_source
    # then inserts; re-find the anchor each time and skip past notes already added after it,
    # so shifting indices don't matter and a group sharing one anchor keeps input order
    added = set()
    for after_id, src in notes:
        pos = next(i for i, c in enumerate(cells) if c.get('id') == after_id) + 1
        while pos < len(cells) and cells[pos].get('id') in added:
            pos += 1
        cell = dict(makers[cell_type](src))
        added.add(cell['id'])
        cells.insert(pos, cell)
    json.dump(nb, open(path, 'w'), indent=1, ensure_ascii=False)
    return f"OK: {path} -- inserted {len(notes)} note(s), patched {len(patches)} cell(s)"

## Smoke test

A quick round-trip against a throwaway notebook -- insert, read, grep, edit, patch -- so a
`nbdev-test` run exercises the whole toolkit end to end. Not exported.

In [ ]:
#| eval: false
# Round-trip smoke test against a throwaway notebook.
import tempfile, os
tmp = tempfile.mktemp(suffix='.ipynb')
_nbf.write(_nbf.v4.new_notebook(cells=[_nbf.v4.new_code_cell('x = 1', id='cell0')]), tmp)

print(insert_cells(tmp, 'cell0', ['y = 2', 'z = 3']))
print(read_nb(tmp))
print(grep_nb(tmp, 'y ='))
print(edit_nb(tmp, 'cell0', 'x = 1', 'x = 100'))
print(patch_nb_cell(tmp, 'cell0', 'x = 999'))

# batch docs: insert two notes after cell0 (order preserved) + rewrite cell0, atomically
print(write_nb_docs(tmp, notes=[['cell0', '# Title'], ['cell0', 'intro paragraph']],
                 patches=[['cell0', 'x = 4242']]))
print(read_nb(tmp, full=True))

os.remove(tmp)